# CIOOS ERDDAP API for CMP Data
Purpose:
- Review the different data formats that can be output by the CIOOS ERDDAP server and how they work with CMP data
- Reference code for using the ERDDAP API to retrieve CMP data

Created on: 2026-09-24

Last updated on: 2026-09-24

## 0 Import libraries

In [1]:
# Library imports
from erddapy import ERDDAP
import os
import pandas as pd
import numpy as np
import xarray as xr
import netCDF4 as nc
import matplotlib.pyplot as plt

## 1 Download the dataset

### 1.1 Initialize ERDDAP instance

In [3]:
e = ERDDAP(
    server = "https://cioosatlantic.ca/erddap",
    protocol = "tabledap"
)

### 1.2 Determine IDs of CMAR datasets

In [14]:
e.dataset_id = 'allDatasets'
e.variables = ['datasetID', 'institution', 'title', 'minTime', 'maxTime']

e.constraints = {'institution=': 'Centre for Marine Applied Research (CMAR)'}
cmar_dataset_summary_df = e.to_pandas()
cmar_dataset_summary_df

,datasetID,institution,title,minTime (UTC),maxTime (UTC)
0,knwz-4bap,Centre for Marine Applied Research (CMAR),Annapolis County Water Quality Data,2020-06-11T19:15:00Z,2023-05-24T17:23:45Z
1,hrs3-qwwh,Centre for Marine Applied Research (CMAR),Annapolis County Wave Data,2020-10-22T19:00:00Z,2020-11-30T10:00:00Z
2,kgdu-nqdp,Centre for Marine Applied Research (CMAR),Antigonish County Water Quality Data,2018-07-24T22:15:00Z,2025-11-19T17:30:58Z
3,5daj-5icy,Centre for Marine Applied Research (CMAR),Cape Breton County Water Quality Data,2016-09-13T16:00:00Z,2025-07-03T16:40:30Z
4,gfri-gzxa,Centre for Marine Applied Research (CMAR),Colchester County Water Quality Data,2020-10-01T14:00:00Z,2025-10-06T16:18:00Z
5,wpsu-7fer,Centre for Marine Applied Research (CMAR),Digby County Water Quality Data,2016-01-21T20:00:00Z,2025-06-03T14:00:00Z
6,7iig-p4yk,Centre for Marine Applied Research (CMAR),Digby County Wave Data,2011-01-06T19:00:00Z,2022-11-19T01:00:00Z
7,eb3n-uxcb,Centre for Marine Applied Research (CMAR),Guysborough County Water Quality Data,2015-07-29T04:00:00Z,2025-11-20T19:59:30Z
8,d5r3-x2vi,Centre for Marine Applied Research (CMAR),Guysborough County Wave Data,2014-07-09T17:07:13Z,2019-04-25T17:00:00Z
9,x9dy-aai9,Centre for Marine Applied Research (CMAR),Halifax County Water Quality Data,2017-07-24T16:45:00Z,2025-12-06T13:53:28Z


### 1.3 Filter to Water Quality datasets

In [15]:
cmar_wq_dataset_summary_df = cmar_dataset_summary_df[cmar_dataset_summary_df['title'].str.contains('Water Quality Data')].copy()
cmar_wq_dataset_summary_df['county'] = cmar_wq_dataset_summary_df['title'].str.removesuffix(' County Water Quality Data')

cmar_wq_dataset_summary_df

,datasetID,institution,title,minTime (UTC),maxTime (UTC),county
0,knwz-4bap,Centre for Marine Applied Research (CMAR),Annapolis County Water Quality Data,2020-06-11T19:15:00Z,2023-05-24T17:23:45Z,Annapolis
2,kgdu-nqdp,Centre for Marine Applied Research (CMAR),Antigonish County Water Quality Data,2018-07-24T22:15:00Z,2025-11-19T17:30:58Z,Antigonish
3,5daj-5icy,Centre for Marine Applied Research (CMAR),Cape Breton County Water Quality Data,2016-09-13T16:00:00Z,2025-07-03T16:40:30Z,Cape Breton
4,gfri-gzxa,Centre for Marine Applied Research (CMAR),Colchester County Water Quality Data,2020-10-01T14:00:00Z,2025-10-06T16:18:00Z,Colchester
5,wpsu-7fer,Centre for Marine Applied Research (CMAR),Digby County Water Quality Data,2016-01-21T20:00:00Z,2025-06-03T14:00:00Z,Digby
7,eb3n-uxcb,Centre for Marine Applied Research (CMAR),Guysborough County Water Quality Data,2015-07-29T04:00:00Z,2025-11-20T19:59:30Z,Guysborough
9,x9dy-aai9,Centre for Marine Applied Research (CMAR),Halifax County Water Quality Data,2017-07-24T16:45:00Z,2025-12-06T13:53:28Z,Halifax
11,a9za-3t63,Centre for Marine Applied Research (CMAR),Inverness County Water Quality Data,2015-11-26T21:20:51Z,2025-11-19T19:30:43Z,Inverness
14,adpu-nyt8,Centre for Marine Applied Research (CMAR),Pictou County Water Quality Data,2017-07-25T20:30:00Z,2025-11-19T14:57:38Z,Pictou
15,qspp-qhb6,Centre for Marine Applied Research (CMAR),Queens County Water Quality Data,2020-06-25T21:21:58Z,2025-07-10T16:00:00Z,Queens


### 1.4 Set desired variables

In [16]:
e.variables = [
 'waterbody',
 'station',
 'deployment_start_date',
 'deployment_end_date',
 'time',
 'depth',
 'temperature',
 'qc_flag_temperature']

### 1.5 (Optional) Set time constraints on data to retrieve

In [19]:
#e.constraints = { "time>=": "2020-09-01", "time<=": "2024-08-31" }
e.constraints = None

### 1.6 Download data and store locally
Avoid waiting for the download every time - takes 15 minutes. [Requesting compressed files](https://www.ncei.noaa.gov/erddap/files/documentation.html#notFtp) is considered good practice to speed up the process.

A couple of examples of data formats are shown below, but many more are available. Potential download formats can be viewed in a drop-down available on the [ERDDAP download form at CIOOS Atlantic](https://cioosatlantic.ca/erddap/tabledap/knwz-4bap.html).

### 1.6.1 Option A: Download as compressed CSV

In [20]:

os.makedirs('data', exist_ok=True)

for index, row in cmar_wq_dataset_summary_df.iterrows():

    csvfile = f"data/{row['county']}.csv.gz"

    if os.path.exists(csvfile):
        continue

    print(f"Downloading {row['title']}...")
    e.dataset_id = row['datasetID'] 
    df = e.to_pandas()

    df.to_csv(csvfile, compression='gzip', index=False)

/home/user/Documents/ssnippets/.venv/lib/python3.14/site-packages/erddapy/core/interfaces.py:36: DtypeWarning: Columns (1: station) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(data, **(pandas_kwargs or {}))


### 1.6.2 Option B: Download as a NetCDF file

In [ ]:

os.makedirs('data', exist_ok=True)

for index, row in cmar_wq_dataset_summary_df.iterrows():

    ncfile = f"data/{row['county']}.nc.gz"

    if os.path.exists(ncfile):
        continue

    print(f"Downloading {row['title']}...")
    e.dataset_id = row['datasetID'] 
    df = e.to_xarray()

    df.to_netcdf(ncfile, mode='w', format='NETCDF4')
    
    from erddapy import ERDDAP
import xarray as xr

# Initialize ERDDAP instance
e = ERDDAP(
    server="https://ioos.us",
    protocol="tabledap",
    response="nc"  # Requesting NetCDF format
)

# Set up your dataset and parameters
e.dataset_id = "whoi_406-20160902T1704"
e.variables = ["depth", "latitude", "longitude", "temperature", "time"]
e.constraints = {
    "time>=": "2016-09-03T00:00:00Z",
    "time<=": "2016-09-04T00:00:00Z",
}

# Open the download URL directly with xarray
url = e.get_download_url()
ds = xr.open_dataset(url)

print(ds)

ReadTimeout: HTTPSConnectionPool(host='cioosatlantic.ca', port=443): Read timed out. (read timeout=60)

### 1.5.3 Option C: Download as a Parquet file